# CZGM garnet growth and diffusion — 1D spherical radial tutorial

**Companion to `CZGM_garnet_model_annulus.ipynb` (2D Cartesian annulus): the same benchmark data, solved on a 1D spherical radial grid.**

This notebook models growth zonation and diffusional relaxation of a spherical
garnet grain with `UWDiffusion.fd_1d.MulticomponentDiffusion1D`, a conservative
finite-volume solver on a 1D radial grid. It is the **spherical 1D** version of
the benchmark: instead of a 2D annulus mesh in the $(x, y)$ plane, the grain is
represented by the radius $r \in [0, R]$, and the diffusion operator carries the
full spherical area factor ($r^2$):

$$
\frac{\partial X_i}{\partial t}
= \frac{1}{r^2}\frac{\partial}{\partial r}
\left(r^2\sum_j D_{ij}\frac{\partial X_j}{\partial r}\right).
$$

**How this differs from the 2D annulus notebook**

| | 2D annulus notebook | this notebook |
|---|---|---|
| geometry | 2D Cartesian annulus mesh (slice through the grain) | 1D spherical radial grid |
| solver | `MulticomponentDiffusionModel` (UW3 finite elements) | `MulticomponentDiffusion1D` (finite volume) |
| dependencies | Underworld3 + PETSc (+ MPI) | `numpy` + `scipy` + `matplotlib` only |
| typical runtime | minutes | seconds |

Because the geometric diffusion operator differs, final profiles are close to —
but not identical with — the 2D annulus result and the CZGM reference output
(see the closing notes).

### CZGM input data

All inputs live in `Tutorials/Diffusion/CZGM_data/` (located automatically
below, wherever the notebook is launched from inside the repository):

- `combined_array.csv` — 8 columns, no header: `R (µm)`, `t (Myr)`, `P (bar)`,
  `T (K)`, `Fe`, `Mg`, `Mn`, `Ca`. The first rows (constant `t`) describe the
  initial core; each later row gives the grain radius, rim composition and
  P–T conditions at one growth step.
- `PTt_path.csv` — 3 columns, no header: `P (bar)`, `T (K)`, `t (Myr)` — the
  prograde-to-retrograde pressure–temperature–time path.
- `combined_C_final.csv` — reference final profiles: `R (µm)`, `Fe`, `Mg`,
  `Mn`, `Ca` (used for comparison at the end, if present).

Fe, Mg and Mn are the independent components; Ca is implicit through the
closure $\sum_k X_k = 1$.

> **Temperature units.** The stored `T` columns are in **Kelvin** (the PTt path
> spans 719.33–883.23 K, i.e. 446–610 °C) and must be used directly in
> Arrhenius expressions — no °C→K conversion. Older versions of this notebook
> mislabelled the column as °C; that labelling is corrected here.

In [ ]:
from pathlib import Path
import sys
import time

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import interpolate

try:
    from UWDiffusion.fd_1d import MulticomponentDiffusion1D, darken_reduced_diffusivity
except ImportError:
    # UWDiffusion is not installed as a package: fall back to the in-repo
    # source tree (src/UWDiffusion) by walking up from the current directory.
    _repo_src = None
    for _base in (Path.cwd(), *Path.cwd().parents):
        if (_base / "src" / "UWDiffusion" / "fd_1d.py").exists():
            _repo_src = _base / "src"
            break
    if _repo_src is None:
        raise ImportError(
            "UWDiffusion is not installed and no src/UWDiffusion/fd_1d.py "
            "was found above the current directory."
        )
    sys.path.insert(0, str(_repo_src))
    from UWDiffusion.fd_1d import MulticomponentDiffusion1D, darken_reduced_diffusivity


In [ ]:
def find_data_dir(start=Path.cwd(), name="CZGM_data", required="combined_array.csv"):
    """Walk up the directory tree until `name` (containing `required`) is found."""
    for base in (start, *start.parents):
        candidate = base / "Tutorials" / "Diffusion" / name
        if (candidate / required).exists():
            return candidate
        candidate = base / name
        if (candidate / required).exists():
            return candidate
    raise FileNotFoundError(
        f"Could not locate a '{name}' directory containing '{required}' above "
        f"{start}. Run this notebook from within the UWDiffusion repository."
    )


DATA_DIR = find_data_dir()
print(f"Data directory: {DATA_DIR}")

In [ ]:
CZGM_data = pd.read_csv(DATA_DIR / "combined_array.csv", header=None)
PTt_path = pd.read_csv(DATA_DIR / "PTt_path.csv", header=None)

# Column order matches the table above: R, t, P, T, Fe, Mg, Mn, Ca
Rr, tr, Pr, Tr, Fer, Mgr, Mnr, Car = [
    np.asarray(CZGM_data.iloc[:, i].to_numpy(dtype=float)) for i in range(8)
]

# Interpolants for P (bar) and T (K — stored Kelvin, used directly) vs t (Myr)
T_of_t = interpolate.interp1d(PTt_path.iloc[:, 2], PTt_path.iloc[:, 1], fill_value="extrapolate")
P_of_t = interpolate.interp1d(PTt_path.iloc[:, 2], PTt_path.iloc[:, 0], fill_value="extrapolate")

print(f"combined_array.csv: {CZGM_data.shape[0]} rows (radius levels), "
      f"{CZGM_data.shape[1]} columns")
print(f"PTt_path.csv:       {PTt_path.shape[0]} rows (path points)")
print(f"grain radius: 0 to {Rr.max():.3f} µm;  growth history: "
      f"{tr.min():.4f} to {tr.max():.4f} Myr")

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4))
ax1.plot(PTt_path.iloc[:, 2], PTt_path.iloc[:, 1], c="tab:red")
ax1.set_xlabel("time (Myr)")
ax1.set_ylabel("T (K)")
ax1.grid(alpha=0.3)
ax2.plot(PTt_path.iloc[:, 2], PTt_path.iloc[:, 0] / 1e3, c="tab:blue")
ax2.set_xlabel("time (Myr)")
ax2.set_ylabel("P (kbar)")
ax2.grid(alpha=0.3)
fig.suptitle("PTt path")
plt.tight_layout()
plt.show()

In [ ]:
# --- User-tunable parameters --------------------------------------------------
SEC_PER_MYR = 1.0e6 * 365.25 * 24.0 * 3600.0   # seconds per Myr (SI time unit)

QUICK_RUN = False                       # True -> coarser grid + larger dt
GRID_STRIDE = 2 if QUICK_RUN else 1     # use every n-th data radius as a vertex
MAX_DT_MYR = 0.1 if QUICK_RUN else 0.05 # maximum time step (Myr)

COMPONENTS = ["Fe", "Mg", "Mn", "Ca"]   # order matches the data columns
DEPENDENT = "Ca"                        # closure: Ca = 1 - Fe - Mg - Mn

DT_S = MAX_DT_MYR * SEC_PER_MYR

In [ ]:
# Rows with t > min(t) describe growth steps: grain radius, rim composition and
# P-T conditions at each step. Earlier rows describe the initial core.
radial_growth = Rr[tr > tr.min()]       # µm
time_evolution = tr[tr > tr.min()]      # Myr
Fe_evolution = Fer[tr > tr.min()]
Mg_evolution = Mgr[tr > tr.min()]
Mn_evolution = Mnr[tr > tr.min()]
Ca_evolution = Car[tr > tr.min()]

initial_profiles = {"Fe": Fer, "Mg": Mgr, "Mn": Mnr, "Ca": Car}
colours = {"Fe": "tab:red", "Mg": "tab:green", "Mn": "tab:blue", "Ca": "gold"}

print(f"initial grain radius: {Rr[tr == tr.min()].max():.3f} µm "
      f"at t = {tr.min():.4f} Myr")
print(f"growth steps: {len(radial_growth)} "
      f"(to {radial_growth[-1]:.3f} µm at t = {time_evolution[-1]:.4f} Myr)")

### Tracer diffusivities (Chakraborty & Ganguly, 1992)

Fe, Mg and Mn tracer diffusivities follow the Arrhenius formulation of
[Chakraborty & Ganguly (1992)](https://doi.org/10.1016/B978-0-08-042718-4.50009-3):

$$
D_i = D_{0,i}\,\exp\!\left(-\frac{E_{a,i} + (P-1)\,E_{v,i}/41.84}{R\,T}\right)
$$

with $D_0$ in cm² s⁻¹, $E_a$ in cal mol⁻¹, $E_v$ in cm³ mol⁻¹ (41.84 converts to
cal), $R = 1.9872$ cal mol⁻¹ K⁻¹, $T$ in K and $P$ in bar. Ca is approximated as
$D_{\mathrm{Ca}} = 0.05\,D_{\mathrm{Mn}}$, following the 2D notebook.

> **Temperature convention (important).** The tabulated `T` column is in
> **Kelvin**, and the CZGM benchmark uses the stored values *directly* as the
> absolute temperature in the Arrhenius expression — the correct thermodynamic
> use of the data. The original MATLAB source subtracts 273 only when handing
> `T` to a Celsius-facing output routine and adds it back inside the Arrhenius
> term, so no conversion belongs in the model itself. Earlier versions of this
> notebook mislabelled the column as °C; adding 273.15 to the stored values
> would double-convert, raise the effective diffusivities by roughly 4–5 orders
> of magnitude, and almost fully homogenize a 76 µm grain over the ~5 Myr
> modelled here.

In [ ]:
def tracer_diffusivities_m2s(temp_K, pressure_bar):
    """Tracer diffusivities (m^2/s) for [Fe, Mg, Mn, Ca] from T (K) and P (bar).

    Chakraborty & Ganguly (1992) parameters. `temp_K` must be thermodynamic
    temperature: the CZGM data files store T in Kelvin and the benchmark uses
    the stored values directly (see the markdown above). Do **not** add 273.15
    here — that would double-convert the stored Kelvin temperatures.
    """
    D0 = {"Fe": 6.36e-4, "Mg": 1.11e-3, "Mn": 5.15e-4}   # cm^2/s
    Ea = {"Fe": 65824.0, "Mg": 67997.0, "Mn": 60569.0}   # cal/mol
    Ev = {"Fe": 5.63, "Mg": 5.27, "Mn": 6.04}            # cm^3/mol
    R_cal = 1.9872                                       # cal/(mol K)

    def _D(elem):
        return D0[elem] * np.exp(
            -(Ea[elem] + (pressure_bar - 1.0) * Ev[elem] / 41.84) / (R_cal * temp_K)
        )

    D_Fe = _D("Fe") * 1.0e-4   # cm^2/s -> m^2/s
    D_Mg = _D("Mg") * 1.0e-4
    D_Mn = _D("Mn") * 1.0e-4
    D_Ca = 0.05 * D_Mn
    return np.array([D_Fe, D_Mg, D_Mn, D_Ca])


# How strongly do the tracer diffusivities vary along the PTt path?
T_path = np.asarray(PTt_path.iloc[:, 1].to_numpy(dtype=float))
P_path = np.asarray(PTt_path.iloc[:, 0].to_numpy(dtype=float))
t_path = np.asarray(PTt_path.iloc[:, 2].to_numpy(dtype=float))
D_path = np.array([tracer_diffusivities_m2s(T, P) for T, P in zip(T_path, P_path)])

fig, ax = plt.subplots(figsize=(8, 4))
for i, comp in enumerate(COMPONENTS):
    ax.semilogy(t_path, D_path[:, i], color=colours[comp], label=comp)
ax.set_xlabel("time (Myr)")
ax.set_ylabel(r"$D$ (m$^2$/s)")
ax.set_title("Tracer diffusivities along the PTt path")
ax.legend(ncols=2)
ax.grid(alpha=0.3, which="both")
plt.show()

### Coupled (inter)diffusion: the Darken / CZGM reduced matrix

Tracer diffusivities are combined into the composition-dependent multicomponent
interdiffusion matrix of the CZGM formulation:

$$
D_{ij} = D_i\,\delta_{ij} - \frac{D_i X_i}{\sum_k D_k X_k}\,(D_j - D_{\mathrm{dep}}),
$$

which `fd_1d.darken_reduced_diffusivity` builds as the reduced
$(N-1)\times(N-1)$ matrix (here $3 \times 3$ for the independent Fe, Mg, Mn,
with Ca dependent). Because the matrix depends on composition **and** on P–T
conditions — hence on model time — it is supplied to the solver as a *callable*.

The 1D solver inspects the callable's signature: naming the first argument `t`
tells the solver that the callable is **time-first**, and it evaluates the
matrix with the model's current solution time at every face of every step. The
composition argument may arrive as a dict of component mole fractions or as an
array in `COMPONENTS` order — the cell below handles both.

In [ ]:
def darken_reduced_diffusivity_callable(t, X):
    """Reduced 3x3 interdiffusion matrix (m^2/s) at model time `t` (s) and `X`.

    The first argument is named `t` so the solver recognises the time-first
    signature and evaluates this callable with the current model time.
    """
    t_myr = float(t) / SEC_PER_MYR
    temp_K = float(T_of_t(t_myr))   # stored Kelvin, used directly
    pressure_bar = float(P_of_t(t_myr))
    D_tracer = tracer_diffusivities_m2s(temp_K, pressure_bar)

    if isinstance(X, dict):
        X_vec = np.array([float(X[name]) for name in COMPONENTS])
    else:
        X_vec = np.asarray(X, dtype=float)

    return darken_reduced_diffusivity(D_tracer, X_vec, dependent_component=3)


# Smoke test: reduced matrix at the start of the growth history (core composition)
X_core = np.array([Fer[0], Mgr[0], Mnr[0], Car[0]])
D_start = darken_reduced_diffusivity_callable(time_evolution[0] * SEC_PER_MYR, X_core)
print(f"reduced interdiffusion matrix (m^2/s) at t = {time_evolution[0]:.4f} Myr:")
print(D_start)

### Building the 1D spherical model

- **Grid**: the data radii `Rr` (0 → 76.12 µm) are used directly as grid
  vertices (converted to metres), so model profiles can be compared pointwise
  with the reference data. `GRID_STRIDE > 1` coarsens the grid for quick runs.
- **Geometry**: `geometry="spherical"` ($\alpha = 2$). The finite-volume scheme
  integrates $r^2$ over each control volume, handles the origin $r = 0$ without
  any singularity, and enforces the symmetry condition $J(0) = 0$ naturally.
- **Initial composition**: uniform core composition (first data row); Ca
  follows from closure. Boundaries start as zero flux at both ends; the growth
  steps below re-pin the rim.
- **Clock**: the model starts at the first growth time, `time_evolution[0]`.

In [ ]:
r_grid_m = Rr[::GRID_STRIDE] * 1.0e-6   # data radii, µm -> m

model = MulticomponentDiffusion1D(
    grid=r_grid_m,
    diffusivity=darken_reduced_diffusivity_callable,
    geometry="spherical",
    components=COMPONENTS,
    dependent_component=DEPENDENT,
    initial_composition={"Fe": Fer[0], "Mg": Mgr[0], "Mn": Mnr[0]},
)

model.reset(time_evolution[0] * SEC_PER_MYR)   # start of the growth history

print(f"grid: {model.n_points} vertices over {model.r[-1] * 1e6:.2f} µm "
      f"({model.geometry}, alpha = {model.alpha})")
print(f"components: {model.components}; dependent: {model.dependent_component}")
print(f"model time starts at {model.time / SEC_PER_MYR:.4f} Myr")

### Phase 1 — prograde growth and diffusion

Each growth step *stamps* the new shell: every grid vertex outside the current
growth radius is set to that step's rim composition (Ca closes automatically),
and the outer boundary is re-pinned to the rim composition as a Dirichlet
condition. This is `apply_growth` — the 1D analogue of the shell-boundary
updates in the 2D notebook. The model then diffuses to the next growth time
with `run_to_time`, while the diffusivity callable samples the PTt path.

In [ ]:
prograde_start = time.time()
for i in range(len(time_evolution) - 1):
    model.apply_growth(
        radial_growth[i] * 1.0e-6,
        {"Fe": Fe_evolution[i], "Mg": Mg_evolution[i], "Mn": Mn_evolution[i]},
        set_rim_bc=True,
    )
    model.run_to_time(time_evolution[i + 1] * SEC_PER_MYR, DT_S)
    print(f"growth step {i + 1:2d}/{len(time_evolution) - 1}: "
          f"r = {radial_growth[i]:6.2f} µm, t = {model.time / SEC_PER_MYR:.4f} Myr")

profile_prograde = {c: model.get_composition(c).copy() for c in COMPONENTS}
print(f"\nprograde done in {time.time() - prograde_start:.1f} s "
      f"(model time {model.time / SEC_PER_MYR:.4f} Myr)")

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(16, 4), sharey=True)
for ax, comp in zip(axes, COMPONENTS):
    ax.plot(Rr, initial_profiles[comp], ls=":", color=colours[comp],
            label="initial (data)")
    ax.plot(model.r * 1e6, profile_prograde[comp], color=colours[comp],
            label="after prograde")
    ax.set_title(comp)
    ax.set_xlabel("r (µm)")
    ax.grid(alpha=0.3)
axes[0].set_ylabel("mole fraction")
axes[0].legend()
plt.tight_layout()
plt.show()

### Phase 2 — final rim and retrograde diffusion

Growth ends at `radial_growth[-1]` = 76.12 µm, the full grid radius. The final
`apply_growth` stamps the last rim composition onto the edge vertex and pins
the rim Dirichlet boundary for the remainder of the run. The model then
diffuses along the retrograde limb of the PTt path — each `PTt_path.csv` entry
starts one segment. No new material is added: this is pure diffusional
relaxation, with the rim acting as a reservoir held at the final growth
composition (so component integrals may drift as the interior re-equilibrates
with the rim — see the check at the end).

In [ ]:
# Final growth step: stamp the last rim composition and pin the rim boundary.
model.apply_growth(
    radial_growth[-1] * 1.0e-6,
    {"Fe": Fe_evolution[-1], "Mg": Mg_evolution[-1], "Mn": Mn_evolution[-1]},
    set_rim_bc=True,
)

# Snapshot before the retrograde stage, for the mass-budget diagnostic below.
mass_before_retro = model.total_mass()

retro_start = time.time()
ptt_times = np.asarray(PTt_path.iloc[:, 2].to_numpy(dtype=float))
retro_times = ptt_times[ptt_times >= time_evolution[-1] - 1.0e-2]

for j in range(len(retro_times) - 1):
    model.run_to_time(retro_times[j + 1] * SEC_PER_MYR, DT_S)

profile_final = {c: model.get_composition(c).copy() for c in COMPONENTS}
print(f"retrograde done in {time.time() - retro_start:.1f} s "
      f"({len(retro_times) - 1} PTt segments; model time "
      f"{model.time / SEC_PER_MYR:.4f} Myr)")

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(16, 4), sharey=True)
for ax, comp in zip(axes, COMPONENTS):
    ax.plot(Rr, initial_profiles[comp], ls=":", color=colours[comp],
            label="initial (data)")
    ax.plot(model.r * 1e6, profile_prograde[comp], ls="--", color=colours[comp],
            label="after prograde")
    ax.plot(model.r * 1e6, profile_final[comp], color=colours[comp],
            label="final (model)")
    ax.set_title(comp)
    ax.set_xlabel("r (µm)")
    ax.grid(alpha=0.3)

# CZGM reference final profiles, where available
reference_file = DATA_DIR / "combined_C_final.csv"
if reference_file.exists():
    reference = pd.read_csv(reference_file, header=None).to_numpy(dtype=float)
    for ax, ref_col in zip(axes, range(1, 5)):
        ax.scatter(reference[:, 0], reference[:, ref_col], marker="x", c="k",
                   label="CZGM reference")
else:
    print(f"Reference file not found ({reference_file}); plotting model output only.")

axes[0].set_ylabel("mole fraction")
axes[0].legend(ncols=2, fontsize=8)
plt.tight_layout()
plt.show()

In [ ]:
# Checks: closure (exact by construction) and the retrograde mass budget.
X_now = model.get_composition()
print(f"max |sum_k X_k - 1| = {np.max(np.abs(X_now.sum(axis=1) - 1.0)):.3e}")

# The rim is a Dirichlet boundary (a reservoir at the final growth
# composition), so during the retrograde stage each component's integral
# changes by the net diffusive flux through the rim. Components richer than
# the rim in the interior (Mn) drain out; poorer ones (Mg, Fe) are supplied.
# This is expected behaviour for a pinned-rim model, not a conservation error.
mass_after_retro = model.total_mass()
print("\nretrograde mass budget (V-weighted spherical integrals):")
for comp in COMPONENTS:
    change = (mass_after_retro[comp] - mass_before_retro[comp]) / mass_before_retro[comp]
    print(f"  {comp}: before = {mass_before_retro[comp]:.6e}, "
          f"after = {mass_after_retro[comp]:.6e}, relative change = {change:+.2e}")

### Summary and notes

- This is the **spherical 1D** counterpart of the 2D Cartesian annulus
  benchmark: the grain is a radial line, and the diffusion operator carries the
  spherical $r^2$ area factor. The 2D annulus notebook instead solves a
  two-dimensional annulus whose out-of-plane contribution differs — so profiles
  track the CZGM reference closely, but are not expected to match the 2D
  annulus result exactly (the same caveat noted in that notebook).
- Growth is applied with `apply_growth` (rim stamping plus a rim Dirichlet
  boundary); time integration uses `run_to_time` with an unconditionally
  stable implicit scheme, so the time-step size is an accuracy, not stability,
  choice.
- The reduced interdiffusion matrix is rebuilt at every face and step by the
  callable `darken_reduced_diffusivity_callable(t, X)` — note the **time-first**
  argument convention, which the solver detects from the parameter name.

**Things to try**

- Set `QUICK_RUN = True` for a fast, coarser preview.
- Interpolate `Rr` onto a finer grid and/or reduce `MAX_DT_MYR` to check
  convergence.
- Switch `geometry="cylindrical"` to model a cylindrical instead of spherical
  grain, and compare against the other reference variants in `CZGM_data/`
  (`combined_C_final-spherical=0.csv`, `combined_C_final_cart.csv`).